In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("..") / "data" / "raw" / "2025q1"   # notebooks run from the notebooks/ folder

# 1. What's in the zip?
for f in sorted(RAW.rglob("*")):
    if f.is_file():
        print(f"{f.relative_to(RAW)}  {f.stat().st_size / 1e6:.1f} MB")

ADMIN.tsv  1.0 MB
ADVISER.tsv  1.8 MB
AUTHORIZED_PARTICIPANT.tsv  2.5 MB
BROKER.tsv  4.6 MB
BROKER_DEALER.tsv  3.0 MB
CCO_EMPLOYER.tsv  0.1 MB
CHIEF_COMPLIANCE_OFFICER.tsv  0.2 MB
COLLATERAL_MANAGER.tsv  0.2 MB
CONTRACT_SECURITY.tsv  0.4 MB
CREDIT_USER.tsv  139.2 MB
CUSTODIAN.tsv  14.3 MB
DEPOSITOR.tsv  0.1 MB
DIRECTOR.tsv  0.6 MB
DIRECTOR_FILE_NUMBER.tsv  2.6 MB
DIVESTMENT.tsv  0.0 MB
ETF.tsv  0.3 MB
FEEDER_FUNDS.tsv  0.0 MB
FOREIGN_INVESTMENT.tsv  0.0 MB
FUND_REPORTED_INFO.tsv  2.1 MB
INTER_FUND_BORROWING_DETAIL.tsv  0.1 MB
INTER_FUND_LENDING_DETAIL.tsv  0.1 MB
LINE_OF_CREDIT_DETAIL.tsv  0.5 MB
LINE_OF_CREDIT_INSTITUTION.tsv  1.5 MB
LOCATION_BOOKS_RECORD.tsv  1.4 MB
MASTER_FUNDS.tsv  0.0 MB
ncen_metadata.json  0.2 MB
ncen_readme.htm  0.7 MB
PRICING_SERVICE.tsv  2.6 MB
PRINCIPAL_TRANSACTION.tsv  3.9 MB
PRINCIPAL_UNDERWRITER.tsv  0.2 MB
PUBLIC_ACCOUNTANT.tsv  0.1 MB
REGISTRANT.tsv  0.4 MB
REGISTRANT_HELDS_SECURITY.tsv  0.0 MB
REGISTRANT_REPORTING_SERIES.tsv  0.1 MB
REGISTRANT_WEBSITE.t

In [2]:
def load(table):
    # find the file whose name (without extension) matches the table name
    matches = [f for f in RAW.rglob("*") if f.is_file() and f.stem.upper() == table]
    assert len(matches) == 1, f"Expected one file for {table}, found {matches}"
    path = matches[0]

    df = pd.read_csv(path, sep="\t", dtype=str, encoding="utf-8",
                 keep_default_na=False, na_values=[""])

    # count raw lines so we can check pandas didn't swallow or split any rows
    with open(path, encoding="utf-8") as fh:
        n_lines = sum(1 for _ in fh) - 1   # minus the header
    print(f"{table}: {len(df):,} rows x {df.shape[1]} cols  (file has {n_lines:,} data lines)")
    return df

sub  = load("SUBMISSION")
reg  = load("REGISTRANT")
fund = load("FUND_REPORTED_INFO")
cust = load("CUSTODIAN")

SUBMISSION: 1,956 rows x 22 cols  (file has 1,956 data lines)
REGISTRANT: 1,956 rows x 36 cols  (file has 1,956 data lines)
FUND_REPORTED_INFO: 6,881 rows x 108 cols  (file has 6,881 data lines)
CUSTODIAN: 86,839 rows x 10 cols  (file has 86,839 data lines)


In [3]:
import sqlite3

DB = Path("..") / "data" / "explore_2025q1.db"

con = sqlite3.connect(DB)
for name, df in {"SUBMISSION": sub, "REGISTRANT": reg,
                 "FUND_REPORTED_INFO": fund, "CUSTODIAN": cust}.items():
    df.to_sql(name, con, if_exists="replace", index=False)
con.close()

def q(sql):
    """Run a query against the exploration database and return a DataFrame."""
    con = sqlite3.connect(DB)
    try:
        return pd.read_sql(sql, con)
    finally:
        con.close()